In [89]:
import numpy as np
from typing import List, Dict, Tuple, Literal
from model_ranking import get_output_paths, load_h5
from model_ranking import get_summary_results, results_to_arrays


In [90]:
source = "BBBC039"
target = "BBBC039"
selected_augmentations: Dict[str, List[str]] = {
    "none": [],
    "DO": ["a005", "a01", "a02", "a03", "a04", "a05"],
}
selected_norms: List[Tuple[float, float]] = [(5, 98)]
source_model="BC_model4"
result_folder = "FP_1"
base_path = "/scratch/talks/patch_segmentation/nuclei"


In [91]:
output_paths = get_output_paths(
    source=source,
    target=target,
    selected_augmentations=selected_augmentations,
    selected_norms=selected_norms,
    source_model=source_model,
    base_dir_path=base_path,
    result_folder=result_folder,
)

In [92]:
output_paths

['/scratch/talks/patch_segmentation/nuclei/BBBC039_to_BBBC039_gap/feature_perturbation_consistency/FP_1/BC_model4/norm_5_98/none/metric_summary.h5',
 '/scratch/talks/patch_segmentation/nuclei/BBBC039_to_BBBC039_gap/feature_perturbation_consistency/FP_1/BC_model4/norm_5_98/DO_a005/metric_summary.h5',
 '/scratch/talks/patch_segmentation/nuclei/BBBC039_to_BBBC039_gap/feature_perturbation_consistency/FP_1/BC_model4/norm_5_98/DO_a01/metric_summary.h5',
 '/scratch/talks/patch_segmentation/nuclei/BBBC039_to_BBBC039_gap/feature_perturbation_consistency/FP_1/BC_model4/norm_5_98/DO_a02/metric_summary.h5',
 '/scratch/talks/patch_segmentation/nuclei/BBBC039_to_BBBC039_gap/feature_perturbation_consistency/FP_1/BC_model4/norm_5_98/DO_a03/metric_summary.h5',
 '/scratch/talks/patch_segmentation/nuclei/BBBC039_to_BBBC039_gap/feature_perturbation_consistency/FP_1/BC_model4/norm_5_98/DO_a04/metric_summary.h5',
 '/scratch/talks/patch_segmentation/nuclei/BBBC039_to_BBBC039_gap/feature_perturbation_consiste

In [93]:
perf_key = "hard_f1"
consis_key = "HD_th05_per_patch"
average_type: Literal["mean", "median"] = "median"
seed = 42
proportion = 1
rng = np.random.default_rng(seed)
sel_inds = None
none_count = 0
pert_count = 0
consis_avg_scores = np.zeros(len(output_paths)-1)
for path in output_paths:
    if "none" in path:
        none_count += 1
        perf_scores = load_h5(path, perf_key)
        if sel_inds is None:
            sel_inds = rng.choice(len(perf_scores), size=int(proportion * len(perf_scores)), replace=False)
        slc_perf = perf_scores[sel_inds]
        if slc_perf.ndim == 2:
            slc_perf = slc_perf[:, 1] 
        if average_type == "mean":
            perf_mean = np.nanmean(slc_perf)
        elif average_type == "median":
            perf_mean = np.nanmedian(slc_perf)
        
    else:
        consis_scores = load_h5(path, consis_key)
        if sel_inds is None:
            sel_inds = rng.choice(len(consis_scores), size=int(proportion * len(consis_scores)), replace=False)
        if average_type == "mean":
            consis_avg_scores[pert_count] = np.nanmean(consis_scores[sel_inds])
        elif average_type == "median":
            consis_avg_scores[pert_count] = np.nanmedian(consis_scores[sel_inds])
        pert_count += 1
assert none_count == 1, "There should be one and only one none path"

In [94]:
print(consis_avg_scores)

[0.00766223 0.01065457 0.01458456 0.01980619 0.02617805 0.03289678]


In [95]:
selected_augmentations: Dict[str, List[str]] = {
    "none": [],
    "DO": [ "a005", "a01", "a02", "a03", "a04", "a05"],
}
source_models = {
    "BBBC039": "BC_model4",
    "DSB2018": "DSB_model4",
    "Go-Nuclear": "GN_model4",
    "HeLaNuc": "HN_model1",
    "Hoechst": "Hst_model5",
    "S_BIAD634": "634_model1",
    "S_BIAD895": "895_model2",
    "S_BIAD1196": "1196_model3",
    "S_BIAD1410": "1410_model2",
}
per_target_norms: Dict[str, List[Tuple[float, float]]] = {
    "BBBC039": [(5, 98)],
    "DSB2018": [(5, 98)],
    "Go-Nuclear": [(0, 99.8)],
    "HeLaNuc": [(5, 99.6)],
    "Hoechst": [(5, 98)],
    "S_BIAD634": [(5, 98)],
    "S_BIAD895": [(5, 98)],
    "S_BIAD1196": [(5, 98)],
    "S_BIAD1410": [(5, 98)],
}

result_folders = {
    "BBBC039": "FP_1",
    "DSB2018": "FP_1",
    "Go-Nuclear": "FP_1",
    "HeLaNuc": "FP_1",
    "Hoechst": "FP_1",
    "S_BIAD634": "FP_1",
    "S_BIAD895": "FP_1",
    "S_BIAD1196": "FP_1",
    "S_BIAD1410": "FP_1",
}
consis_keys = {
    "BBBC039": "HD_th05",
    "DSB2018": "HD_th05",
    "Go-Nuclear": "HD_th05",
    "HeLaNuc": "HD_th05",
    "Hoechst": "HD_th05",
    "S_BIAD634": "HD_th05",
    "S_BIAD895": "HD_th05",
    "S_BIAD1196": "HD_th05",
    "S_BIAD1410": "HD_th05",
}
target_datasets = [
    "BBBC039",
    #"DSB2018",
    #"Hoechst",
    #"S_BIAD634",
    #"S_BIAD895",
]
source_datasets = [
    "BBBC039",
    #"DSB2018",
    #"Go-Nuclear",
    #"HeLaNuc",
    #"Hoechst",
    #"S_BIAD634",
    #"S_BIAD895",
    #"S_BIAD1196",
    #"S_BIAD1410",
]
base_seg_dir = "/scratch/talks/patch_segmentation/nuclei"

In [96]:
for i, target in enumerate(target_datasets):
    if (target == "S_BIAD895") and (target in source_datasets):
        sources = source_datasets.copy()
        sources.remove(target)
    else:
        sources = source_datasets
    consis_str, perf_str, NA_perf = get_summary_results(
        source_data=sources, 
        target_data=[target], 
        source_models=source_models, 
        selected_augmentations=selected_augmentations,
        selected_norms=per_target_norms,
        consis_keys=consis_keys,
        perf_key="hard_f1",
        per_target_norms=True,
        result_folders=result_folders,
        approach="feature_perturbation_consistency",
        consis_postfix= "median_per_alpha",
        perf_postfix="median",
        base_seg_dir=base_seg_dir,
    )
    consis_scores, NA_perf_scores = results_to_arrays(
        consis_str, NA_perf, "DO", len(selected_augmentations["DO"])
    )

Source: BBBC039


100%|██████████| 1/1 [00:00<00:00, 25.68it/s]


In [97]:
for key in consis_str.keys():
    print(consis_str[key])

{'norm_5_98': {'DO': array([0.00766223, 0.01065457, 0.01458456, 0.01980619, 0.02617805,
       0.03289678])}}


In [98]:
print(consis_scores)

[[0.00766223 0.01065457 0.01458456 0.01980619 0.02617805 0.03289678]]


In [100]:
print(consis_avg_scores)

[0.00766223 0.01065457 0.01458456 0.01980619 0.02617805 0.03289678]


In [88]:
print(NA_perf_scores)

[0.96826011]
